In [ ]:
source("Main.R")
source("Utilities.R")
source("Conf.R")
library(ArchR)
library(parallel)
library(BSgenome)
library(BSgenome.Hsapiens.UCSC.hg38)
library(universalmotif)
library(TFBSTools)
library(motifmatchr)


In [ ]:
# addArchRThreads(threads = 90) 
# addArchRGenome("hg38")
# myProj = loadArchRProject(path = "/data/Osaka_Omics/scATAC/OsakaMomicsATAC", force = FALSE, showLogo = TRUE)
# myProj <- addMotifAnnotations(ArchRProj = myProj, motifSet = "cisbp", name = "Motif")
# #saveArchRProject(ArchRProj = myProj, outputDirectory = paste0(atacFilesDir ,"OsakaMomicsATAC"), 
# #load = FALSE)
#getAvailableMatrices(myProj)
# myMotifs = getMatches(ArchRProj = myProj, name = "Motif")
# kk = as.matrix(assay(myMotifs))
# kk[kk==FALSE] = 0
# kk[kk==TRUE] = 1
# rownames(kk) = rownames(k)
# write.csv(kk, "/data/Osaka_Omics/scATAC/OsakaMomicsATAC/DiffPeaks/CSVFiles/MotifMatrix_cisbp.csv")

In [ ]:
motifMat = read.csv("/data/Osaka_Omics/scATAC/OsakaMomicsATAC/DiffPeaks/CSVFiles/MotifMatrixTFGroups_v2.csv", row.names=1)
peak_strings = rownames(motifMat)
split_peaks <- do.call(rbind, strsplit(peak_strings, "_"))
chr <- split_peaks[, 1]
start <- as.numeric(split_peaks[, 2])
end <- as.numeric(split_peaks[, 3])
gr <- GRanges(seqnames = chr, ranges = IRanges(start = start, end = end))


In [ ]:
file_path="HOCOMOCOv11_core_pwms_HUMAN_mono.txt"
lines <- readLines(file_path)
split_indices <- grep("^>", lines)
split_indices <- c(split_indices, length(lines) + 1)

pwm_list <- list()

for (i in seq_along(split_indices[-length(split_indices)])) {
start <- split_indices[i]
end <- split_indices[i + 1] - 1

motif_name <- gsub("^>", "", lines[start])
pwm_lines <- lines[(start + 1):end]

pwm_matrix <- do.call(rbind, lapply(pwm_lines, function(line) {
  vals <- as.numeric(strsplit(trimws(line), "\t")[[1]])
  if (length(vals) != 4) stop("Each row must have exactly 4 values")
  return(vals)
}))

colnames(pwm_matrix) <- c("A", "C", "G", "T")

if (!is.null(pwm_matrix)) {
  pwm_obj <- tryCatch({
    TFBSTools::PWMatrix(
      ID = motif_name,
      name = motif_name,
      profileMatrix = t(as.matrix((pwm_matrix))),
      bg = c(A = 0.25, C = 0.25, G = 0.25, T = 0.25)
    )
  }, error = function(e) {
    warning(sprintf("Failed to convert motif %s: %s", motif_name, e$message))
    return(NULL)
  })

  if (inherits(pwm_obj, "PWMatrix")) {
    pwm_list[[motif_name]] <- pwm_obj
  }
}
}
valid_pwms <- Filter(function(x) inherits(x, "PWMatrix"), pwm_list)
#return(TFBSTools::PWMatrixList(valid_pwms))
pwm_list_obj <- do.call(TFBSTools::PWMatrixList, valid_pwms)
pwm_list_obj

In [ ]:
motif_matches <- matchMotifs(pwm_list_obj, subject = gr, genome = "hg38", out = "matches")
match_df <- as.data.frame(as.matrix(assay(motif_matches)))


In [ ]:
colnames(match_df) <- sapply(colnames(match_df), function(x){strsplit(x,"_")[[1]][1]})

In [ ]:
match_df_num = data.frame(lapply(match_df, function(x){as.numeric(x)}))

In [ ]:
 rownames(gr)

In [ ]:
rownames(match_df_num) = rownames(motifMat)
write.csv(match_df_num,
          "/data/Osaka_Omics/scATAC/OsakaMomicsATAC/DiffPeaks/CSVFiles/MotifMatrixTFGroups_hocomoco.csv")


In [ ]:
read_multiple_pwms_as_PWMatrixList <- function(file_path) {
  lines <- readLines(file_path)
  split_indices <- grep("^>", lines)
  split_indices <- c(split_indices, length(lines) + 1)

  pwm_list <- list()

  for (i in seq_along(split_indices[-length(split_indices)])) {
    start <- split_indices[i]
    end <- split_indices[i + 1] - 1

    motif_name <- gsub("^>", "", lines[start])
    pwm_lines <- lines[(start + 1):end]

    pwm_matrix <- do.call(rbind, lapply(pwm_lines, function(line) {
      vals <- as.numeric(strsplit(trimws(line), "\t")[[1]])
      if (length(vals) != 4) stop("Each row must have exactly 4 values")
      return(vals)
    }))

    colnames(pwm_matrix) <- c("A", "C", "G", "T")

    if (!is.null(pwm_matrix)) {
      pwm_obj <- tryCatch({
        TFBSTools::PWMatrix(
          ID = motif_name,
          name = motif_name,
          profileMatrix = t(as.matrix((pwm_matrix))),
          bg = c(A = 0.25, C = 0.25, G = 0.25, T = 0.25)
        )
      }, error = function(e) {
        warning(sprintf("Failed to convert motif %s: %s", motif_name, e$message))
        return(NULL)
      })

      if (inherits(pwm_obj, "PWMatrix")) {
        pwm_list[[motif_name]] <- pwm_obj
      }
    }
  }

  valid_pwms <- Filter(function(x) inherits(x, "PWMatrix"), pwm_list)
  return(TFBSTools::PWMatrixList(valid_pwms))

}



In [ ]:
PWMatrixList = read_multiple_pwms_as_PWMatrixList("HOCOMOCOv11_core_pwms_HUMAN_mono.txt")

In [ ]:
read_meme_to_PWMatrixList <- function(file_path) {
  lines <- readLines(file_path)
  
  motif_blocks <- grep("^MOTIF ", lines)
  if (length(motif_blocks) == 0) stop("No MOTIF found in MEME file.")

  pwm_list <- list()
  bg_line <- grep("^Background letter frequencies", lines)
  if (length(bg_line) > 0) {
    bg_lines <- lines[(bg_line + 1):(bg_line + 1)]
    bg_vals <- unlist(strsplit(bg_lines, "\\s+"))
    bg_vals <- bg_vals[bg_vals != ""]
    bg <- as.numeric(bg_vals[seq(2, length(bg_vals), by = 2)])
    names(bg) <- bg_vals[seq(1, length(bg_vals), by = 2)]
  } else {
    bg <- c(A = 0.25, C = 0.25, G = 0.25, T = 0.25)
  }

  for (i in seq_along(motif_blocks)) {
    motif_start <- motif_blocks[i]
    motif_end <- if (i < length(motif_blocks)) motif_blocks[i + 1] - 1 else length(lines)
    block <- lines[motif_start:motif_end]

    # Extract motif ID and name
    motif_header <- strsplit(block[1], "\\s+")[[1]]
    motif_id <- motif_header[2]
    motif_name <- if (length(motif_header) > 2) motif_header[3] else motif_id

    # Find matrix start line
    mat_start <- grep("^letter-probability matrix", block)
    if (length(mat_start) == 0) next

    mat_lines <- block[(mat_start + 1):length(block)]
    mat_lines <- mat_lines[grepl("^[ \t]*[0-9eE.+\\-]", mat_lines)]
    mat <- do.call(rbind, lapply(mat_lines, function(line) as.numeric(strsplit(trimws(line), "[ \t]+")[[1]])))

    if (ncol(mat) != 4) next

    colnames(mat) <- c("A", "C", "G", "T")
    mat <- t(mat)

    pwm <- TFBSTools::PWMatrix(ID = motif_id, name = motif_name, profileMatrix = as.matrix(mat), bg = bg)
    pwm_list[[motif_name]] <- pwm
  }

  pwm_list_obj <- do.call(TFBSTools::PWMatrixList, pwm_list)

}

pwm_list_obj = read_meme_to_PWMatrixList("HOCOMOCOv9.meme")

In [ ]:
pwm_list_obj

In [ ]:
library(TFBSTools)

generatePWMatrixObject <- function(pwmFile){
        # Step 1: Read all lines
        lines <- readLines(pwmFile)
        
        # Step 2: Extract motif name
        motif_name <- sub("^>", "", lines[1])
        
        # Step 3: Read the matrix data (skip first line)
        mat <- read.table(text = lines[-1], header = FALSE, sep = "\t", stringsAsFactors = FALSE)
        
        # Step 4: Transpose so rows are A, C, G, T
        mat_t <- t(as.matrix(mat))  # Ensures numeric matrix
        
        # Step 5: Assign row names to be DNA bases (IMPORTANT!)
        rownames(mat_t) <- c("A", "C", "G", "T")
        
        # Step 6: Create PWMatrix object
        pwm <- PWMatrix(ID = motif_name,
                        name = motif_name,
                        profileMatrix = mat_t)
        return(pwm)
}



In [ ]:
motif_RUNX1 = generatePWMatrixObject("RUNX1_HUMAN.pwm")
motif_RUNX3 = generatePWMatrixObject("RUNX3_HUMAN.pwm")
motifs <- PWMatrixList(motif_RUNX1, motif_RUNX3)
motif_matches <- matchMotifs(motifs, subject = gr, genome = "hg38", out = "matches")
match_df <- as.data.frame(as.matrix(assay(motif_matches)))
colnames(match_df) = c("RUNX1","RUNX3")
table(match_df)

In [ ]:
match_df$RUNX1 = as.numeric(match_df$RUNX1)
match_df$RUNX3 = as.numeric(match_df$RUNX3)

In [ ]:
table(match_df)

In [ ]:
table(motifMat[,c("RUNX1","RUNX3")])

In [ ]:
motifMat$RUNX1 = match_df$RUNX1
motifMat$RUNX3 = match_df$RUNX3

In [ ]:
table(motifMat[,c("RUNX1","RUNX3")])

In [ ]:
write.csv(motifMat, 
          "/data/Osaka_Omics/scATAC/OsakaMomicsATAC/DiffPeaks/CSVFiles/MotifMatrixTFGroups_v3.csv",
          row.names=FALSE)